# Extension – Phonetic Lesk

In Task 6 we added the phonetic similarity to Wu-Palmer, i.e. we compared the **names of two senses** by their sound, and it did not help: two senses do not have a similar meaning because their names sound similar. Here we use the phonetic similarity where it makes more sense: inside **Lesk**, to compare **words**.

Lesk counts the words that the context and the definition of a sense have in common, but only if they are written in exactly the same way. Medical texts often write the same word in different ways: British and American spelling (*haemorrhage* / *hemorrhage*, *tumour* / *tumor*), singular and plural (*eyelid* / *eyelids*), small typos. With a phonetic or edit-distance comparison, these words can count as the same word.

In [ ]:
import sys
sys.path.append("..")                    
import utils as u                             

import re
import pandas as pd
from nltk.corpus import stopwords
from wsd_utils import (ALL_ACRONYMS, ALL_TERMS, MAPPING, wordnet_senses, sample_instances,
                       accuracy, mean)

STOPWORDS = set(stopwords.words("english"))

**Three ways to compare two words**
- **exact**: the two words are identical (standard Lesk);
- **Soundex**: the two words have the same Soundex code (Task 5);
- **Levenshtein**: the two words differ by at most one letter. We apply it only to words of at least 5 letters, because between short words one letter changes everything (e.g. *milk* / *mild*).

In [2]:
def same_exact(a, b):
    return a == b


def same_soundex(a, b):
    return u.soundex(a) == u.soundex(b)


def same_levenshtein(a, b, max_dist=1, min_len=5):
    if a == b:
        return True
    if min(len(a), len(b)) < min_len or abs(len(a) - len(b)) > max_dist:
        return False
    return u.levenshtein(a, b) <= max_dist


MATCHES = {"exact": same_exact, "Soundex": same_soundex, "Levenshtein": same_levenshtein}

pairs = [("haemorrhage", "hemorrhage"), ("oedema", "edema"), ("tumour", "tumor"), ("eyelid", "eyelids"),
         ("drug", "dirge"), ("heart", "hurt"), ("teeth", "tooth"), ("milk", "mild"), ("cell", "cello")]
pd.DataFrame([{"pair": f"{a} / {b}", **{name: f(a, b) for name, f in MATCHES.items()}} for a, b in pairs]).set_index("pair")

,exact,Soundex,Levenshtein
pair,,,
haemorrhage / hemorrhage,False,True,True
oedema / edema,False,False,True
tumour / tumor,False,True,True
eyelid / eyelids,False,False,True
drug / dirge,False,True,False
heart / hurt,False,True,False
teeth / tooth,False,True,False
milk / mild,False,False,False
cell / cello,False,True,False


**Observations:** Levenshtein recognizes all four spelling variants and does not join any pair of different words. Soundex fails in both directions: it joins different words (*drug* / *dirge*, *heart* / *hurt*, *teeth* / *tooth*, *cell* / *cello*), because it keeps only the first letter and the consonants, and it misses some variants (*oedema* / *edema* start with a different letter, and even the plural *eyelids* changes the code). So we expect Soundex to add wrong matches to Lesk, and Levenshtein to help.

**Lesk with a word comparison**

We use our own simplified Lesk from Task 2: the signature of a sense is its WordNet definition, examples and synonyms, without stopwords and without the target word. The score of a sense is the number of context words that **match** at least one word of its signature, where "match" is one of the three comparisons above. The context is the whole abstract, as in Task 2.

In [3]:
from functools import lru_cache
from wsd_utils import CONCEPT_MAP


def tokenize(text):
    """Lowercase words of a text, without stopwords and short words."""
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOPWORDS and len(w) > 2}


def signature(sense, target):
    """Words describing a sense: definition + examples + synonyms, without the target word."""
    text = " ".join([sense.definition()] + sense.examples() + [l.replace("_", " ") for l in sense.lemma_names()])
    return tokenize(text) - tokenize(target)


def overlap(context, sig, match):
    """Number of context words that match at least one word of the signature."""
    return sum(any(match(c, s) for s in sig) for c in context)


def phonetic_lesk(context, word, match):
    """Sense of the word whose signature has the largest overlap with the context."""
    senses = wordnet_senses(word)
    if not senses:
        return None
    return max(senses, key=lambda s: overlap(context, signature(s, word), match)).name()


# make Soundex faster: each word is coded only once
u.soundex = lru_cache(maxsize=None)(u.soundex)

In [6]:
inst = sample_instances("Cilia")[-1]                   # an abstract about eyelashes
context = tokenize(inst["context"]) - tokenize("Cilia")
print("Correct sense:", MAPPING["Cilia"][inst["cui"]])
for name, match in MATCHES.items():
    print(f"{name:12} -> {phonetic_lesk(context, 'Cilia', match)}")

Correct sense: ['eyelash.n.01']
exact        -> cilium.n.01
Soundex      -> cilium.n.01
Levenshtein  -> cilium.n.01


In [7]:
def matched_words(context, sig, match):
    """Context words that match at least one word of the signature."""
    return sorted(c for c in context if any(match(c, s) for s in sig))


print(inst["context"][:300], "...\n")
for name, match in MATCHES.items():
    print(name)
    for s in wordnet_senses("Cilia"):
        words = matched_words(context, signature(s, "Cilia"), match)
        print(f"   {s.name():15} {len(words)}  {words}")

Ectopic cilia--a rare anomaly. ...

exact
   cilium.n.01     0  []
   eyelash.n.01    0  []
Soundex
   cilium.n.01     0  []
   eyelash.n.01    0  []
Levenshtein
   cilium.n.01     0  []
   eyelash.n.01    0  []


**Observation:** this abstract has only its title, so with exact matching no sense shares any word and Lesk takes the first WordNet sense. Soundex gives one point to *cilium* because the target word *cilia* sounds like *cell* (C400) in its definition: a wrong match. So, from now on, we also remove the target word from the **context**, not only from the signatures.

**Evaluation**

We run the three versions on our 39 words, with the same sample of abstracts as Tasks 4 and 6 (10 per sense), and compute accuracy and accuracy on covered senses with our manual mapping, as in Task 4.

In [8]:
import time

WORDS = [w for w in ALL_ACRONYMS + ALL_TERMS if w in MAPPING]
ACRONYMS = [w for w in WORDS if w in ALL_ACRONYMS]
TERMS = [w for w in WORDS if w in ALL_TERMS]

start = time.time()
predictions = {}
for word in WORDS:
    rows = []
    for inst in sample_instances(word):
        context = tokenize(inst["context"]) - tokenize(word)          # without the target word
        answers = {name: phonetic_lesk(context, word, match) for name, match in MATCHES.items()}
        rows.append({"cui": inst["cui"], "answers": answers})
    predictions[word] = rows
print(f"{sum(len(r) for r in predictions.values())} abstracts in {time.time() - start:.0f} s")

table = []
for name in MATCHES:
    line = {"comparison": name}
    for group, words in [("acronyms", ACRONYMS), ("terms", TERMS)]:
        line[f"{group} acc"] = mean(accuracy(predictions[w], w, name) for w in words)
        line[f"{group} cov"] = mean(accuracy(predictions[w], w, name, covered_only=True) for w in words)
    table.append(line)

results = pd.DataFrame(table).set_index("comparison").round(3)
results

810 abstracts in 31 s


,acronyms acc,acronyms cov,terms acc,terms cov
comparison,,,,
exact,0.255,0.875,0.429,0.656
Soundex,0.242,0.820,0.403,0.621
Levenshtein,0.252,0.865,0.439,0.668


**Observations**
- **Soundex makes Lesk worse** on both groups (e.g. terms 0.429 → 0.403): its wrong matches (*cilia* / *cell*, *drug* / *dirge*) give points to wrong senses more often than it finds real spelling variants.
- **Levenshtein slightly improves terms** (0.429 → 0.439, covered 0.656 → 0.668) and is almost the same on acronyms. The effect is small: spelling variants exist in the abstracts, but they are not frequent enough to change many answers.
- This agrees with Task 6: the Soundex code is too coarse to compare words (or senses), while a careful edit distance can help a little.
- Note: our exact Lesk on the whole abstract (0.429 on terms) is better than all the pywsd methods of Task 4 (best 0.371), but the comparison is not fair: Task 4 used a window of 10 words, and our signatures also contain examples and synonyms.

In [9]:
changes = []
for word in WORDS:
    for r in predictions[word]:
        e, l = r["answers"]["exact"], r["answers"]["Levenshtein"]
        if e != l:
            gold = MAPPING[word][r["cui"]]
            changes.append({"word": word, "exact": e, "Levenshtein": l,
                            "effect": "better" if l in gold and e not in gold else
                                      "worse" if e in gold and l not in gold else "both wrong"})

changes = pd.DataFrame(changes)
print(changes["effect"].value_counts(), "\n")
changes.groupby(["word", "effect"]).size().unstack(fill_value=0)

effect
both wrong    20
better        10
worse          8
Name: count, dtype: int64 



effect,better,both wrong,worse
word,,,
ALS,0,1,0
CAD,0,1,0
CAM,0,2,0
Ca,0,1,2
Callus,1,0,1
Cell,1,0,0
Cement,0,0,1
DON,0,1,0
ERUPTION,2,0,1


**Where Levenshtein changes the answer**
- Levenshtein changes the prediction in only 38 of 810 abstracts: 10 better, 8 worse, 20 wrong in both cases.
- Most "both wrong" cases are acronyms (GAG, RA, CAM, US): their correct senses are poorly covered by WordNet, so a different match cannot help.
- On terms the effect is positive (+4): Respiration (+2), Exercises (+2), ERUPTION and Cell (+1); only Nurse and Cement lose one abstract.
- Interestingly, Respiration and ERUPTION were the words that got worse with the Soundex similarity in Task 6, while here they improve: the idea of an approximate match can help, but only if it is precise.